In [0]:
dbutils.widgets.text("catalog_name", "dbw_lakehouse_dev")
dbutils.widgets.text("silver_schema", "cloud_practice_silver")
dbutils.widgets.text("gold_schema", "cloud_practice_gold")

catalog_name = dbutils.widgets.get("catalog_name")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

silver_orders = f"{catalog_name}.{silver_schema}.orders"
silver_customers = f"{catalog_name}.{silver_schema}.customers"
silver_lines = f"{catalog_name}.{silver_schema}.order_lines"

gold_order_summary = f"{catalog_name}.{gold_schema}.order_summary"
gold_revenue_by_country = f"{catalog_name}.{gold_schema}.revenue_by_country"

print("Silver:", silver_orders, silver_customers, silver_lines)
print("Gold:", gold_order_summary, gold_revenue_by_country)

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}
COMMENT 'Gold layer - business-ready aggregates and summaries'
""")

In [0]:
from pyspark.sql.functions import current_timestamp

order_summary_df = spark.sql(
    f"""
    SELECT
      o.order_id,
      o.ordered_at,
      o.status,
      o.customer_id,
      c.name AS customer_name,
      c.country,
      COUNT(l.line_id) AS line_count,
      ROUND(SUM(l.line_total), 2) AS order_total,
      current_timestamp() AS gold_loaded_at
    FROM {silver_lines} l
    INNER JOIN {silver_orders} o ON l.order_id = o.order_id
    INNER JOIN {silver_customers} c ON o.customer_id = c.customer_id
    GROUP BY
      o.order_id,
      o.ordered_at,
      o.status,
      o.customer_id,
      c.name,
      c.country
    """
)

revenue_by_country_df = spark.sql(
    f"""
    SELECT
      c.country,
      COUNT(DISTINCT o.order_id) AS order_count,
      COUNT(DISTINCT c.customer_id) AS customer_count,
      ROUND(SUM(l.line_total), 2) AS revenue,
      current_timestamp() AS gold_loaded_at
    FROM {silver_lines} l
    INNER JOIN {silver_orders} o ON l.order_id = o.order_id
    INNER JOIN {silver_customers} c ON o.customer_id = c.customer_id
    GROUP BY c.country
    """
)

print("order_summary rows:", order_summary_df.count())
print("revenue_by_country rows:", revenue_by_country_df.count())
display(order_summary_df)
display(revenue_by_country_df)

In [0]:
from delta.tables import DeltaTable

def merge_table(target_table: str, source_df, merge_key: str) -> None:
    if spark.catalog.tableExists(target_table):
        (
            DeltaTable.forName(spark, target_table)
            .alias("t")
            .merge(source_df.alias("s"), f"t.{merge_key} = s.{merge_key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        source_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

# Grain = order_id → MERGE (ADF-safe incremental)
merge_table(gold_order_summary, order_summary_df, "order_id")

# Grain = country → small aggregate → full refresh each run
(
    revenue_by_country_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable(gold_revenue_by_country)
)

print("Gold load completed.")

In [0]:
display(spark.table(gold_order_summary).orderBy("ordered_at"))
display(spark.table(gold_revenue_by_country).orderBy("revenue", ascending=False))